In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns",50)

**1. Load the Dataset**

titanic dataset

In [ ]:
df = pd.read_csv("titanic_dataset.csv")
df.shape
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


**2. Adding few columns for demonstrating Data Issues**

In [ ]:
#ID Like column
df["passanger_id"] = np.arange(1, len(df)+1)

#Constant Column
df["constant_col"] = 1

In [22]:
df["embarked_town_dirty"] = df["embark_town"].copy

sample_idx = df["embarked_town_dirty"].dropna().sample(6,random_state = 42).index

df.loc[sample_idx[0:2] , "embarked_town_dirty"] = df.loc[sample_idx[0:2],"embarked_town_dirty"].str.upper()
df.loc[sample_idx[2:4] , "embarked_town_dirty"] = df.loc[sample_idx[0:2],"embarked_town_dirty"].str.strip().apply(lambda x: f"{x}")
df.loc[sample_idx[4:6] , "embarked_town_dirty"] = "unknown"

sample_idx

Index([709, 439, 840, 720, 39, 290], dtype='int64')

In [31]:
df = pd.read_csv("titanic_dataset.csv")
df.shape
df.head()
#ID Like column
df["passanger_id"] = np.arange(1, len(df)+1)

#Constant Column
df["constant_col"] = 1

df["embarked_town_dirty"] = df["embark_town"]

sample_idx = df["embarked_town_dirty"].dropna().sample(6,random_state = 42).index

df.loc[sample_idx[0:2] , "embarked_town_dirty"] = df.loc[sample_idx[0:2],"embarked_town_dirty"].str.upper()
df.loc[sample_idx[2:4] , "embarked_town_dirty"] = df.loc[sample_idx[0:2],"embarked_town_dirty"].str.strip().apply(lambda x: f"{x}")
df.loc[sample_idx[4:6] , "embarked_town_dirty"] = "unknown"

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,passanger_id,constant_col,embarked_town_dirty
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,1,1,Southampton
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,1,Cherbourg
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,3,1,Southampton
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,4,1,Southampton
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,5,1,Southampton


**3. Data Quality Checklist**

1. Basic dataset overview
2. Missing values summary
3. Duplicates
4. Data type validation
5. Constant and quasi constant columns
6. ID like columns
7. String inconsistencies
8. High null columns
9. High zero columns(for numeric features)

***3.1 Basic Dataset Overview***

In [36]:
df.nunique()

survived                 2
pclass                   3
sex                      2
age                     88
sibsp                    7
parch                    7
fare                   248
embarked                 3
class                    3
who                      3
adult_male               2
deck                     7
embark_town              3
alive                    2
alone                    2
passanger_id           891
constant_col             1
embarked_town_dirty      5
dtype: int64

In [38]:
df.describe()

,survived,pclass,age,sibsp,parch,fare,passanger_id,constant_col
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000,891.0
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208,446.000000,1.0
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429,257.353842,0.0
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000,1.000000,1.0
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400,223.500000,1.0
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200,446.000000,1.0
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000,668.500000,1.0
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200,891.000000,1.0


***3.2 Missing values Summary***

In [40]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame(
    {
        "missing_count" : missing_count,
        "missing_percent" : missing_percent
    }
)

print("Missing values summary")
missing_summary

Missing values summary


,missing_count,missing_percent
deck,688,77.216611
age,177,19.865320
embarked_town_dirty,4,0.448934
embarked,2,0.224467
embark_town,2,0.224467
survived,0,0.000000
pclass,0,0.000000
parch,0,0.000000
sibsp,0,0.000000
sex,0,0.000000


***3.3 Duplicates***

In [ ]:
duplicates_mask = df.duplicated()
num_duplicates = duplicates_mask.sum()

print("Number of Duplicates:",num_duplicates)

#If you want to remove duplicates
#df_no_duplicates = df.drop_duplicates()

Number of Duplicates: 0


***3.4 Data Type Validation***

In [46]:
expected_types = {
    "survived":"int64",
    "pclass" : "int64",
    "sex" : "category",
    "age" : "float64",
    "fare" : "float64",
    "embark_town" : "category",
    "passanger_id" : "int64"
}

print("Data Type Validation")
for col, expected in expected_types.items():
    if col in df.columns:
        actual = df[col].dtype
        print(f"{col}: actual = {actual}, expected = {expected}")


Data Type Validation
survived: actual = int64, expected = int64
pclass: actual = int64, expected = int64
sex: actual = str, expected = category
age: actual = float64, expected = float64
fare: actual = float64, expected = float64
embark_town: actual = str, expected = category
passanger_id: actual = int64, expected = int64


***3.5 Constand and Quasi - Constant Columns***

In [49]:
n_rows = len(df)
nunique = df.nunique()

constant_cols = nunique[nunique == 1].index.to_list()
print("Constant Columns:",constant_cols)

#Quasi - Constant COlumns

quasi_constant_cols = []

for col in df.columns:
    top_freq = df[col].value_counts(normalize = True,dropna = False).values[0]
    if top_freq > 0.95 and col not in constant_cols:
        quasi_constant_cols.append(col)

print("Quasi - Constant Columns(Top value more than 95%):",quasi_constant_cols)

Constant Columns: ['constant_col']
Quasi - Constant Columns(Top value more than 95%): []


***3.6 ID Like Columns***

In [50]:
n_rows = len(df)

id_like_cols = []

for col in df.columns:
    if df[col].nunique(dropna = False) == n_rows:
        id_like_cols.append(col)

print("ID like columns:",id_like_cols)

ID like columns: ['passanger_id']


***3.7 String Inconsistencies***

In [ ]:
object_cols =df.select_dtypes(include = ["object","string"]).columns.tolist()
print(object_cols)

df["embarked_town_clean"] = (
    df["embark_town_dirty"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("unknown",np.nan)
)

['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive', 'embarked_town_dirty']


***3.8 High Null Columns***

In [58]:
high_null_threshold = 0.4
high_null_cols = missing_summary[missing_summary["missing_percent"] >= high_null_threshold * 100]

print("Columns with high missing percentage:\n",high_null_cols)

Columns with high missing percentage:
       missing_count  missing_percent
deck            688        77.216611


***3.9 High Zero Columns(Numeric)***

In [61]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
zero_share = {}

for col in numeric_cols:
    zero_share[col] = (df[col] == 0).mean()

zero_share_series = pd.Series(zero_share).sort_values(ascending=False)
print(zero_share_series)

print("-"*80)

high_zero_threshold = 0.8
high_zero_cols = zero_share_series[zero_share_series >= high_zero_threshold]
print("Numeric columns with many zeroes",high_zero_cols)

parch           0.760943
sibsp           0.682379
survived        0.616162
fare            0.016835
age             0.000000
pclass          0.000000
passanger_id    0.000000
constant_col    0.000000
dtype: float64
--------------------------------------------------------------------------------
Numeric columns with many zeroes Series([], dtype: float64)
